# A donde va el turista cuando desaparece la VUT

Un piso turistico cierra en 2028. Sus cuatro huespedes necesitan alojamiento. **A cual de los 762
hoteles van?**

Dos cosas les importan: que este **cerca** de donde pensaban quedarse y que **cueste parecido**.
Cual pesa mas es lo que no sabemos, y este cuaderno prueba las tres maneras de resolverlo.

| | Que hace | Veredicto |
|---|---|---|
| **1. Peso manual** | Una barra 0-100% entre precio y cercania | **La que se usa** |
| **2. Regresion** | Estima el peso desde la demanda actual de Airbnb | Descartada: no funciona, y el cuaderno ensena por que |
| **3. Las dos** | La regresion pone el valor inicial, la barra lo mueve | Descartada: hereda el fallo de la 2 |

Y encima de todo eso, **los hoteles no estan vacios**: la ultima parte mete la ocupacion real.

In [1]:
# 1. Carga
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
GOLD = RAIZ / "data" / "gold"

vut = pd.read_csv(GOLD / "airbnb_para_web.csv", low_memory=False)
hoteles = pd.read_csv(GOLD / "alojamientos_reglados.csv", low_memory=False)
hoteles = hoteles[hoteles["banda_plaza"].notna() & hoteles["lat"].notna()].reset_index(drop=True)

print(f"VUT en Airbnb        {len(vut):>7,}  plazas {int(vut['accommodates'].sum()):>7,}")
print(f"Alojamiento reglado  {len(hoteles):>7,}  plazas {int(hoteles['plazas'].sum()):>7,}")

VUT en Airbnb          6,834  plazas  30,067
Alojamiento reglado      762  plazas  84,314


## De donde salen estas cifras, y por que no son las que se leen por ahi

**84.314 plazas regladas.** Cuadra con las fuentes independientes: el INE estima 86.822 para junio
de 2026 e Idescat 83.454 para 2025. Comprobado tambien por dentro --el Hotel Arts declara 1.012
plazas en 494 habitaciones, 2,05 por habitacion--.

**30.067 plazas VUT, no las 61.899 del registro.** El registro oficial tiene 24.075 licencias con
61.899 plazas, pero aqui solo entran las que **se anuncian hoy en Airbnb** y sobreviven la criba:
6.834 anuncios. La diferencia son licencias que estan en otras plataformas, alquiladas por
temporada o paradas.

Son dos cifras distintas y las dos son ciertas. Esta mide la oferta anunciada; la otra, la oferta
legal existente.

In [2]:
# 2. Variables comunes: precio por plaza y distancia
CENTRO = (41.3870, 2.1700)   # Placa de Catalunya
LAT0 = 41.39


def distancia_km(lat, lon, centro=CENTRO):
    """Haversine. Con geometria plana el eje este-oeste sale inflado un tercio a esta latitud."""
    lat1, lon1 = np.radians(centro)
    lat2, lon2 = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def proyectar(lat, lon):
    return np.c_[np.asarray(lon, float) * 111.320 * np.cos(np.radians(LAT0)),
                 np.asarray(lat, float) * 110.570]


vut["precio"] = vut["precio_plaza_anual"]
hoteles["precio"] = hoteles["precio_plaza"]
vut["dist_centro"] = distancia_km(vut["latitude"], vut["longitude"])

# 6.834 x 762 = 5,2 millones de pares. Cabe en memoria.
D = np.sqrt(((proyectar(vut["latitude"], vut["longitude"])[:, None, :]
              - proyectar(hoteles["lat"], hoteles["lon"])[None, :, :]) ** 2).sum(axis=2))
BRECHA = np.abs(vut["precio"].to_numpy()[:, None] - hoteles["precio"].to_numpy()[None, :])

# Las dos magnitudes se llevan a 0-1 para poder mezclarlas. 1 es el mejor hotel posible.
cercania = 1 - D / D.max()
parecido = 1 - BRECHA / BRECHA.max()

print(f"precio mediano   VUT {vut['precio'].median():>6.1f}   hotel {hoteles['precio'].median():>6.1f} EUR/plaza")
print(f"distancia de una VUT al hotel mas cercano: mediana {np.median(D.min(axis=1)):.2f} km")

precio mediano   VUT   51.8   hotel   77.8 EUR/plaza
distancia de una VUT al hotel mas cercano: mediana 0.10 km


# OPCION 1 --- La barra

    utilidad = w x cercania + (1 - w) x parecido_de_precio

`w = 1`: solo importa quedarse cerca. `w = 0`: solo importa pagar lo mismo.

**`w` no se estima: lo pone quien mira la web.** Es deliberado. Un turista aleman y uno andaluz no
tienen la misma sensibilidad al precio, y ningun dato que tengamos distingue entre los dos.

In [3]:
# 3. El reparto, sin limite de aforo todavia
def repartir(w: float) -> pd.DataFrame:
    """Cada VUT lleva sus plazas al hotel de mayor utilidad. Sin mirar si cabe."""
    destino = (w * cercania + (1 - w) * parecido).argmax(axis=1)
    return pd.DataFrame({
        "vut_idx": np.arange(len(vut)),
        "hotel_idx": destino,
        "plazas": vut["accommodates"].to_numpy(),
        "km": D[np.arange(len(vut)), destino],
        "salto_precio": hoteles["precio"].to_numpy()[destino] - vut["precio"].to_numpy(),
    })


filas = []
for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = repartir(w)
    filas.append({"w": w,
                  "km medianos": round(float(r["km"].median()), 2),
                  "sobrecoste EUR/plaza": round(float(r["salto_precio"].median()), 1),
                  "hoteles usados": r["hotel_idx"].nunique()})
display(pd.DataFrame(filas).set_index("w"))

,km medianos,sobrecoste EUR/plaza,hoteles usados
w,,,
0.00,2.22,0.0,581
0.25,0.33,1.1,634
0.50,0.21,4.0,619
0.75,0.14,9.4,618
1.00,0.10,25.6,650


In [4]:
# 4. Que decide de verdad la eleccion
#
# Con w=0,5 las dos notas pesan igual sobre el papel. En la practica no: casi todos los hoteles
# cuestan parecido --mediana 78 EUR/plaza-- asi que entre los candidatos buenos el precio apenas
# los distingue, y la distancia si.
mas_cercano = D.argmin(axis=1)
mas_parecido = BRECHA.argmin(axis=1)

print(f"{'w':>6} {'acaba en el mas CERCANO':>26} {'en el mas PARECIDO en precio':>30}")
for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
    elegido = (w * cercania + (1 - w) * parecido).argmax(axis=1)
    print(f"{w:>6} {np.mean(elegido == mas_cercano):>25.0%} {np.mean(elegido == mas_parecido):>29.0%}")
print()
print("Con w=0,5 solo 2 de cada 100 van al hotel de precio mas parecido, y 25 al mas cercano.")
print("La barra no reparte la influencia al 50%: hay que decirlo en la web.")

     w    acaba en el mas CERCANO   en el mas PARECIDO en precio
   0.0                        0%                          100%
  0.25                       12%                            5%
   0.5                       25%                            2%
  0.75                       48%                            1%


   1.0                      100%                            0%

Con w=0,5 solo 2 de cada 100 van al hotel de precio mas parecido, y 25 al mas cercano.
La barra no reparte la influencia al 50%: hay que decirlo en la web.


# OPCION 2 --- La regresion, y por que se descarta

La idea era no inventarse `w`: mirar la demanda **de hoy** en Airbnb y ver si mandan los pisos
baratos o los centricos. La variable de demanda es `number_of_reviews_ltm`, resenas del ultimo ano,
que hay en los 6.834 anuncios sin un solo nulo.

Los hoteles no tienen ninguna columna de demanda, asi que la regresion solo se puede ajustar sobre
el lado Airbnb.

In [5]:
# 5. La regresion de demanda
import statsmodels.api as sm

d = vut[(vut["number_of_reviews_ltm"] > 0) & (vut["precio"] > 0)].copy()
d["log_demanda"] = np.log(d["number_of_reviews_ltm"])
d["log_precio"] = np.log(d["precio"])

X = d[["log_precio", "dist_centro", "accommodates"]].copy()
X["entera"] = (d["room_type"] == "Entire home/apt").astype(int)
modelo = sm.OLS(d["log_demanda"], sm.add_constant(X)).fit()

print(f"n = {len(d):,}   R2 = {modelo.rsquared:.3f}")
display(pd.DataFrame({"coef": modelo.params.round(4), "p": modelo.pvalues.round(4)}))

print("DOS PROBLEMAS, y ninguno tiene arreglo:")
print()
print(f"1. La distancia al centro no predice nada: p = {modelo.pvalues['dist_centro']:.2f}")
solo_d = sm.OLS(d["log_demanda"], sm.add_constant(d[["dist_centro"]])).fit().rsquared
print(f"   ella sola explica un R2 de {solo_d:.3f}. No es que pese poco: es que no hay senal.")
print()
print(f"2. El precio sale POSITIVO: +1% de precio, {modelo.params['log_precio']:+.2f}% de demanda.")
print("   Eso no es una curva de demanda, es causalidad inversa: el piso bueno cobra mas Y ademas")
print("   se llena. Para medir sensibilidad al precio haria falta ver al mismo piso a dos precios")
print("   distintos, y el volcado es una foto de un solo dia.")

n = 6,834   R2 = 0.122


,coef,p
const,-1.5663,0.0000
log_precio,0.7860,0.0000
dist_centro,0.0097,0.4827
accommodates,0.0825,0.0000
entera,0.7591,0.0000


DOS PROBLEMAS, y ninguno tiene arreglo:

1. La distancia al centro no predice nada: p = 0.48
   ella sola explica un R2 de 0.001. No es que pese poco: es que no hay senal.

2. El precio sale POSITIVO: +1% de precio, +0.79% de demanda.
   Eso no es una curva de demanda, es causalidad inversa: el piso bueno cobra mas Y ademas
   se llena. Para medir sensibilidad al precio haria falta ver al mismo piso a dos precios
   distintos, y el volcado es una foto de un solo dia.


In [6]:
# 6. No es colinealidad, que era la sospecha
#
# Se esperaba que precio y distancia se solaparan --el centro es caro porque es el centro-- y que
# eso impidiera separarlos. El dato dice que no: el problema es otro.
from statsmodels.stats.outliers_influence import variance_inflation_factor

print(f"correlacion precio-distancia: {d['log_precio'].corr(d['dist_centro']):.3f}")
base = sm.add_constant(d[["log_precio", "dist_centro", "accommodates"]])
print("VIF (por encima de 5 habria solapamiento serio):")
for i, v in enumerate(base.columns):
    if v != "const":
        print(f"  {v:<14} {variance_inflation_factor(base.values, i):.2f}")
print()
print("Todos por debajo de 1,1. No se solapan. La regresion falla por las dos razones de arriba.")

correlacion precio-distancia: -0.131
VIF (por encima de 5 habria solapamiento serio):
  log_precio     1.09
  dist_centro    1.03
  accommodates   1.08

Todos por debajo de 1,1. No se solapan. La regresion falla por las dos razones de arriba.


# OPCION 3 --- Descartada

Consistia en que la barra arrancase en el valor de la opcion 2 en vez de en un 50% arbitrario.
**Hereda el fallo:** el valor inicial saldria de un modelo cuyo coeficiente de distancia no es
significativo y cuyo coeficiente de precio tiene el signo cambiado.

Un numero de partida que parece medido y no lo esta es peor que no tener ninguno.

---

# La ocupacion: los hoteles no estan vacios

Todo lo anterior supone 84.314 plazas libres. **No lo estan.**

El INE publica el grado de ocupacion por plazas de Barcelona, y esta en
`data/bronze/serie_ine_barcelona.csv`.

**Se usa la media de los ultimos doce meses, no el mes a mes.** No por comodidad: los precios de
este proyecto son **equivalentes anuales** --el raspado de hoteles se corrigio de estacionalidad y
el de Airbnb tambien--, asi que cruzarlos con una ocupacion mensual mezclaria dos escalas de tiempo
distintas y el resultado no querria decir nada.

In [7]:
# 7. Cuanto sitio hay de verdad
serie = pd.read_csv(RAIZ / "data" / "bronze" / "serie_ine_barcelona.csv", low_memory=False)
ocupacion = serie[serie["serie"].str.contains("Grado de ocupaci", na=False)
                  & serie["serie"].str.contains("por plazas. B", na=False)].sort_values("mes")
ultimos = ocupacion.tail(12)
OCUPACION = float(ultimos["valor"].mean()) / 100

print(f"Ocupacion por plazas, media de {ultimos['mes'].iloc[0]} a {ultimos['mes'].iloc[-1]}: "
      f"{OCUPACION:.1%}")
print(f"  mes mas flojo: {ultimos.loc[ultimos.valor.idxmin(), 'mes']} {ultimos.valor.min():.1f}%")
print(f"  mes mas lleno: {ultimos.loc[ultimos.valor.idxmax(), 'mes']} {ultimos.valor.max():.1f}%")
print()

plazas_totales = hoteles["plazas"].sum()
libres = plazas_totales * (1 - OCUPACION)
necesarias = vut["accommodates"].sum()

print(f"plazas regladas     {plazas_totales:>9,.0f}")
print(f"ocupadas hoy        {plazas_totales * OCUPACION:>9,.0f}")
print(f"LIBRES              {libres:>9,.0f}")
print(f"plazas VUT a mover  {necesarias:>9,.0f}")
print()
print(f"{'NO CABEN: faltan' if libres < necesarias else 'Caben, sobran'} "
      f"{abs(libres - necesarias):,.0f} plazas")

# La ocupacion se reparte por igual entre todos los hoteles. El INE no la publica por
# establecimiento ni por categoria, asi que suponer que un cinco estrellas y una pension estan
# igual de llenos es un supuesto, no un dato.
hoteles["plazas_libres"] = hoteles["plazas"] * (1 - OCUPACION)

Ocupacion por plazas, media de 2025-07 a 2026-06: 67.9%
  mes mas flojo: 2025-11 55.6%
  mes mas lleno: 2025-07 79.1%

plazas regladas        84,314
ocupadas hoy           57,222
LIBRES                 27,092
plazas VUT a mover     30,067

NO CABEN: faltan 2,975 plazas


In [8]:
# 8. El reparto con aforo real
def repartir_con_aforo(w: float, capacidad: np.ndarray) -> tuple:
    """Las VUT eligen por turnos, de mas cara a mas barata. `capacidad` se va gastando."""
    utilidad = w * cercania + (1 - w) * parecido
    orden_hoteles = np.argsort(-utilidad, axis=1)
    turno = np.argsort(-vut["precio"].to_numpy())

    libre = capacidad.astype(float).copy()
    filas = []
    for i in turno:
        pendientes = float(vut["accommodates"].iloc[i])
        for j in orden_hoteles[i]:
            if pendientes <= 0:
                break
            if libre[j] <= 0:
                continue
            cabe = min(pendientes, libre[j])
            libre[j] -= cabe
            pendientes -= cabe
            filas.append({"vut_idx": i, "hotel_idx": j, "plazas": cabe, "km": D[i, j],
                          "salto_precio": hoteles["precio"].iloc[j] - vut["precio"].iloc[i]})
        if pendientes > 0:
            filas.append({"vut_idx": i, "hotel_idx": -1, "plazas": pendientes,
                          "km": np.nan, "salto_precio": np.nan})
    return pd.DataFrame(filas), libre


resultados = []
for etiqueta, cap in [("hoteles vacios (irreal)", hoteles["plazas"].to_numpy()),
                      (f"ocupacion real {OCUPACION:.0%}", hoteles["plazas_libres"].to_numpy())]:
    for w, nombre_w in [(1.0, "quiero mi barrio"), (0.0, "quiero mi precio")]:
        a, _ = repartir_con_aforo(w, cap)
        ok = a[a.hotel_idx >= 0]
        fuera = a.loc[a.hotel_idx < 0, "plazas"].sum()
        resultados.append({
            "aforo": etiqueta, "barra": nombre_w,
            "plazas colocadas": int(ok["plazas"].sum()),
            "SIN SITIO": int(fuera),
            "km medianos": round(float(ok["km"].median()), 2),
            "sobrecoste EUR": round(float(ok["salto_precio"].median()), 1),
        })
display(pd.DataFrame(resultados).set_index(["aforo", "barra"]))

plazas colocadas  SIN SITIO  \
aforo                   barra                                           
hoteles vacios (irreal) quiero mi barrio             30067          0   
                        quiero mi precio             30067          0   
ocupacion real 68%      quiero mi barrio             27092       2974   
                        quiero mi precio             27092       2974   

                                          km medianos  sobrecoste EUR  
aforo                   barra                                          
hoteles vacios (irreal) quiero mi barrio         0.15            29.0  
                        quiero mi precio         2.39             0.1  
ocupacion real 68%      quiero mi barrio         0.35            33.4  
                        quiero mi precio         2.32            22.8

In [9]:
# 9. Quien se queda fuera
a, libre = repartir_con_aforo(1.0, hoteles["plazas_libres"].to_numpy())
sin_sitio = a[a.hotel_idx < 0]
ok = a[a.hotel_idx >= 0]

print(f"=== CON LA BARRA EN 'QUIERO MI BARRIO' Y OCUPACION DEL {OCUPACION:.0%} ===")
print(f"  plazas sin sitio: {sin_sitio['plazas'].sum():,.0f} "
      f"({sin_sitio['plazas'].sum() / vut['accommodates'].sum():.0%} del total)")
print(f"  hoteles llenos  : {int((libre <= 0.01).sum()):,} de {len(hoteles):,}")
print()

# Como las caras eligen primero, quien se queda fuera es quien menos paga.
sin_sitio = sin_sitio.assign(banda=vut["banda_plaza"].to_numpy()[sin_sitio.vut_idx.to_numpy()],
                             barrio=vut["neighbourhood"].to_numpy()[sin_sitio.vut_idx.to_numpy()])
print("Por banda de precio de la VUT:")
print(sin_sitio.groupby("banda")["plazas"].sum().reindex(["€", "€€", "€€€", "€€€€"]).fillna(0)
      .astype(int).to_string())
print()
print("Barrios de origen de quien se queda sin sitio:")
display(sin_sitio.groupby("barrio")["plazas"].sum().sort_values(ascending=False).head(8)
        .astype(int).to_frame("plazas sin sitio"))

=== CON LA BARRA EN 'QUIERO MI BARRIO' Y OCUPACION DEL 68% ===
  plazas sin sitio: 2,975 (10% del total)
  hoteles llenos  : 762 de 762

Por banda de precio de la VUT:
banda
€       2974
€€         0
€€€        0
€€€€       0

Barrios de origen de quien se queda sin sitio:


,plazas sin sitio
barrio,
el Raval,270
la Dreta de l'Eixample,267
"Sant Pere, Santa Caterina i la Ribera",218
Sant Antoni,199
el Barri Gòtic,197
la Nova Esquerra de l'Eixample,185
el Poble Sec,169
l'Antiga Esquerra de l'Eixample,137


# Lo que va a la web

**La barra, con dos posiciones que cuentan la historia**, y la ocupacion real siempre puesta.

Lo que hay que explicar al lado del mapa, sin excepcion:

1. **`w` no se ha medido.** Un turista aleman y uno andaluz no tienen la misma sensibilidad al
   precio, y ningun dato disponible los distingue.
2. **La barra no reparte la influencia al 50% cuando esta en el medio.** La celda 4 lo mide: con
   `w = 0,5` mandan mas los kilometros que los euros, porque casi todos los hoteles cuestan
   parecido.
3. **La ocupacion es la media de los ultimos doce meses**, para que case con unos precios que
   tambien son equivalentes anuales. En agosto el resultado seria mucho peor y en noviembre mucho
   mejor.
4. **Se supone que todos los hoteles estan igual de llenos.** El INE no publica la ocupacion por
   establecimiento ni por categoria.
5. **Son las plazas anunciadas en Airbnb, no las 61.899 del registro.**